# 03 — Pipeline Spark Streaming + Kafka

Este notebook documenta e implementa el flujo en vivo del proyecto:

`datos historicos NYC Taxi -> Kafka -> Spark Streaming -> agregacion por zona/30 min -> modelo GBT -> predicciones`

La idea es simular tiempo real usando datos historicos. Si se emiten los mismos viajes historicos con los mismos timestamps, las features temporales y espaciales son las mismas, asi que las predicciones deben ser reproducibles independientemente de si hoy es lunes o jueves. El dia actual de ejecucion no entra en el modelo; entra el `pickup_datetime` del mensaje historico.

## Rutas usadas

- Kafka local: `kafka_2.12-3.7.0/`
- Topic: `taxi-trips`
- Productor: `producer/taxi_producer.py`
- Consumidor streaming: `streaming/taxi_consumer.py`
- Modelo usado por la guia: `models/gbt_taxi`
- Salida streaming: `streaming_output/`
- Checkpoint Spark: `streaming_checkpoint/`

El modelo `models/gbt_taxi` espera estas features:

```text
zone_lon, zone_lat, hour, dayofweek,
is_weekend, is_rush_hour, is_late_night,
lag_1, lag_2
```

Por eso el consumidor streaming genera esas columnas antes de llamar a `modelo.transform(...)`.

# Guía de arranque del pipeline de streaming

Entorno: contenedor Linux Ubuntu 24 de la universidad (`big26`). Java ya está instalado.

---

## 1. Instalación (solo la primera vez)

```bash
# Descargar Kafka 3.7.0
wget https://archive.apache.org/dist/kafka/3.7.0/kafka_2.12-3.7.0.tgz

# Descomprimir en el home
tar -xzf kafka_2.12-3.7.0.tgz

# Instalar el cliente Python de Kafka
pip install kafka-python --break-system-packages
```

Zookeeper viene incluido dentro de Kafka, no hay que instalarlo por separado.

---

## 2. Preparar los datos de test (solo la primera vez)

Desde la raíz del repo:

```bash
python producer/prepare_stream_test_data.py
```

Esto limpia los datos raw de `data/` y guarda viajes individuales limpios en `data_stream/test_trips/`. Tarda unos minutos.

---

## 3. Arranque del pipeline

Necesitas **3 terminales** abiertas en la raíz del repo.

### Terminal 1 — Kafka y Zookeeper

```bash
bash start_kafka.sh
```

Espera a ver:
```
Zookeeper PID: XXXX | Kafka PID: XXXX
```

Verifica que funciona:
```bash
~/kafka_2.12-3.7.0/bin/kafka-topics.sh --list --bootstrap-server localhost:9092
```

Si no devuelve error, Kafka está listo.

### Terminal 2 — Consumidor Spark Streaming

```bash
python streaming/taxi_consumer.py --starting-offsets latest
```

Espera a ver:
```
Stream de demanda arrancado.
```

### Terminal 3 — Productor Kafka

```bash
python producer/taxi_producer.py --limit 0 --sleep 0
```

A partir de aquí el consumidor empieza a recibir viajes, agruparlos en ventanas de 15 minutos, calcular las variables del modelo y predecir la demanda por zona.

---

## 4. Salida esperada

Cada batch el consumidor imprime:

```
[Batch 6] Ventanas/zona procesadas: 3,001 | MAE batch: 6.46
+-------------------+--------+--------+----------+------------+
|window_start       |zone_lon|zone_lat|trip_count|prediction  |
+-------------------+--------+--------+----------+------------+
...
```

Las predicciones completas se guardan en `streaming_output/` en formato parquet.

---

## 5. Parar el pipeline

- **Productor y consumidor**: `Ctrl+C` en sus respectivas terminales.
- **Kafka y Zookeeper**: los PIDs se imprimen al arrancar con `start_kafka.sh`. Para pararlos:

```bash
kill <KAFKA_PID> <ZOOKEEPER_PID>
```

O de forma más directa:

```bash
~/kafka_2.12-3.7.0/bin/kafka-server-stop.sh
~/kafka_2.12-3.7.0/bin/zookeeper-server-stop.sh
```

---

## Notas

- Kafka y Zookeeper **no persisten entre sesiones**. Hay que arrancarlos cada vez que se abre el contenedor.
- El productor lee de `data_stream/test_trips/`. Si esa carpeta no existe, ejecuta primero el paso 2.
- Los logs de Kafka y Zookeeper quedan en `/tmp/kafka.log` y `/tmp/zookeeper.log` por si hay que depurar.

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
df = spark.read.parquet("/home/alumno/Desktop/bigdata-nyc-taxi/streaming_output/")
df.printSchema()
print(f"Filas: {df.count()}")
df.select("window_start").distinct().orderBy("window_start").show(5, truncate=False)
print(f"Ventanas distintas: {df.select('window_start').distinct().count()}")

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- zone_lon: double (nullable = true)
 |-- zone_lat: double (nullable = true)
 |-- trip_count: double (nullable = true)
 |-- prediction: double (nullable = true)
 |-- error: double (nullable = true)
 |-- abs_error: double (nullable = true)

Filas: 63592
+-------------------+
|window_start       |
+-------------------+
|2009-01-26 23:00:00|
|2009-01-26 23:15:00|
|2009-01-26 23:30:00|
|2009-01-26 23:45:00|
|2009-01-27 00:00:00|
+-------------------+
only showing top 5 rows

Ventanas distintas: 480


## Leer resultados guardados

Cuando el consumidor haya escrito datos en `streaming_output/`, puedes leerlos desde Spark:

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi-streaming-results")
    .master("local[*]")
    .getOrCreate()
)

df_resultados = spark.read.parquet("../streaming_output")
df_resultados.printSchema()
df_resultados.orderBy("window_start", "prediction", ascending=[True, False]).show(20, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 17:08:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/07 17:08:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- zone_lon: double (nullable = true)
 |-- zone_lat: double (nullable = true)
 |-- trip_count: double (nullable = true)
 |-- prediction: double (nullable = true)
 |-- error: double (nullable = true)
 |-- abs_error: double (nullable = true)



+-------------------+-------------------+--------+--------+----------+------------------+-------------------+------------------+
|window_start       |window_end         |zone_lon|zone_lat|trip_count|prediction        |error              |abs_error         |
+-------------------+-------------------+--------+--------+----------+------------------+-------------------+------------------+
|2009-01-26 23:00:00|2009-01-26 23:15:00|-73.98  |40.75   |191.0     |297.07927692303826|106.07927692303826 |106.07927692303826|
|2009-01-26 23:00:00|2009-01-26 23:15:00|-73.99  |40.75   |165.0     |243.90314944207458|78.90314944207458  |78.90314944207458 |
|2009-01-26 23:00:00|2009-01-26 23:15:00|-73.99  |40.76   |159.0     |241.5381630215074 |82.53816302150739  |82.53816302150739 |
|2009-01-26 23:00:00|2009-01-26 23:15:00|-73.98  |40.76   |110.0     |185.13922319596756|75.13922319596756  |75.13922319596756 |
|2009-01-26 23:00:00|2009-01-26 23:15:00|-73.99  |40.74   |131.0     |180.93291061391395|49.93291

26/05/07 17:08:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

# R2
evaluator_r2 = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(df_resultados)

# RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(df_resultados)

# MAE
evaluator_mae = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="mae"
)

mae = evaluator_mae.evaluate(df_resultados)

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

R²: 0.9004
RMSE: 46.3611
MAE: 13.5129


## Version resumida del consumidor

El script real esta en `streaming/taxi_consumer.py`. Esta celda deja visible la logica principal dentro del notebook para que el flujo quede documentado.

In [ ]:
# Resumen conceptual del consumidor:
# 1. Leer mensajes JSON desde Kafka.
# 2. Convertir pickup_datetime a timestamp.
# 3. Filtrar coordenadas validas de NYC.
# 4. Crear zone_lon y zone_lat con grid de 0.01 grados.
# 5. Agregar en streaming por window(pickup_datetime, '30 minutes') + zona.
# 6. Usar outputMode('update') para ver ventanas que se van actualizando.
# 7. Crear hour, dayofweek, is_weekend, is_rush_hour, is_late_night.
# 8. Crear lag_1, lag_2 y lag_48 usando historial por zona.
# 9. Cargar models/gbt_taxi con PipelineModel.load(...).
# 10. Aplicar modelo.transform(features_df).
# 11. Mostrar top zonas y guardar en streaming_output/.

## Si algo falla

- Si aparece `Could not find class org.apache.spark.sql.kafka...`, ejecuta otra vez el consumidor con internet; Spark necesita descargar el paquete Kafka.
- Si cambias el codigo del consumidor o quieres empezar una demo limpia:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi
rm -rf streaming_checkpoint streaming_output
```

- Si el topic tiene mensajes antiguos y quieres reiniciar la demo limpia:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/kafka-topics.sh --delete --topic taxi-trips --bootstrap-server localhost:9092
bin/kafka-topics.sh --create --topic taxi-trips --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
```

- Si por accidente lanzaste primero el productor y despues el consumidor, puedes repetir la demo limpia borrando el topic o relanzando el productor con el consumidor ya abierto. No es el flujo recomendado para la presentacion.
